# RTX-KG2 queries — minimal all-Soufflé pipeline

## Setup

In [1]:
import os
import json
import subprocess
import time
import pandas as pd
from dotenv import load_dotenv
import anthropic

pd.set_option("display.max_colwidth", 80)

NODES_TSV = "rtx-kg2_nodes_2.8.1.tsv"
EDGES_TSV = "rtx-kg2_edges_2.8.1.tsv"
PREDICATES_TSV = "distinct_core_predicate.tsv"
OUT_DIR = "souffle_out"

EDGE_HEADER = (
    ".decl edge(subj:symbol, pred:symbol, obj:symbol, neg:symbol)\n"
    '.input edge(IO=file, filename="' + EDGES_TSV + '", columns="16:0:17:3")'
)

load_dotenv()
client = anthropic.Anthropic(api_key=os.environ["APIKEY"])
MODEL = "claude-sonnet-4-5"

def llm(system, user, max_tokens=1024):
    msg = client.messages.create(model=MODEL, max_tokens=max_tokens,
                                 system=system, messages=[{"role": "user", "content": user}])
    return msg.content[0].text

def _json(text):
    i, j = text.find("{"), text.rfind("}")
    return json.loads(text[i:j + 1])

def _souffle_code(text):
    t = text.strip()
    if t.startswith("```"):
        t = t.split("\n", 1)[1].rsplit("```", 1)[0]
    return t.strip()

KG2_PREDICATES = (pd.read_csv(PREDICATES_TSV, sep="\t")
                  .sort_values("n", ascending=False)["predicate"].astype(str).tolist())
print(str(len(KG2_PREDICATES)) + " predicates")

61 predicates


## 1. Run Soufflé

In [2]:
def _profile_summary(prof_path, wall):
    if not os.path.exists(prof_path):
        return
    rel = json.load(open(prof_path))["root"]["program"]["relation"]
    compute = 0.0
    for name, info in sorted(rel.items()):
        rt = info.get("runtime", {})
        dur = (rt.get("end", 0) - rt.get("start", 0)) / 1e6
        compute += dur
        if dur >= 0.01:
            print("    compute " + name + ": " + format(dur, ".2f") + "s (" + str(info.get("num-tuples")) + " tuples)")
    print("    load+overhead: " + format(max(0.0, wall - compute), ".2f") + "s")

def run_souffle(program, out_name="result"):
    os.makedirs(OUT_DIR, exist_ok=True)
    dl_path = out_name + ".dl"
    with open(dl_path, "w") as f:
        f.write(program + "\n")
    prof = os.path.join(OUT_DIR, out_name + "_profile.json")
    cmd = "ulimit -s unlimited 2>/dev/null || ulimit -s 1048576; exec souffle " + dl_path + " -F . -D " + OUT_DIR + " -p " + prof
    t0 = time.perf_counter()
    subprocess.run(["bash", "-c", cmd], check=True, env=dict(os.environ, OMP_STACKSIZE="1G"))
    wall = time.perf_counter() - t0
    print("[souffle:" + out_name + "] wall " + format(wall, ".1f") + "s")
    _profile_summary(prof, wall)
    path = os.path.join(OUT_DIR, out_name + ".csv")
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return pd.DataFrame()
    return pd.read_csv(path, sep="\t", header=None, dtype=str).fillna("")

## 2. Retrieve node and predicate IDs

In [3]:
def _ci_regex(term):
    parts = []
    for ch in term.strip():
        if ch.isalpha():
            parts.append("[" + ch.lower() + ch.upper() + "]")
        elif ch in '"]^\\':
            parts.append(".")
        else:
            parts.append("[" + ch + "]")
    return ".*" + "".join(parts) + ".*"

def _rank(name, src, term_lc):
    nl = name.lower()
    if nl == term_lc:
        return 0
    if nl.startswith(term_lc):
        return 1
    if term_lc in nl:
        return 2
    return 3

def _search_nodes(term, limit=25):
    pat = _ci_regex(term)
    program = "\n".join([
        ".decl node(id:symbol, name:symbol, category:symbol, syn:symbol)",
        '.input node(IO=file, filename="' + NODES_TSV + '", columns="7:10:15:13")',
        ".decl cand(id:symbol, name:symbol, category:symbol, src:symbol)",
        'cand(i, n, c, "name") :- node(i, n, c, _), match("' + pat + '", n).',
        'cand(i, n, c, "syn") :- node(i, n, c, s), match("' + pat + '", s).',
        ".output cand",
    ])
    df = run_souffle(program, "cand")
    if len(df) == 0:
        return pd.DataFrame(columns=["id", "name", "category"])
    df.columns = ["id", "name", "category", "src"]
    tl = term.strip().lower()
    df["rank"] = [_rank(n, s, tl) for n, s in zip(df["name"], df["src"])]
    df["nlen"] = df["name"].str.len()
    df = df.sort_values(["rank", "nlen"]).drop_duplicates("id").head(limit)
    out = df[["id", "name", "category"]].reset_index(drop=True)
    print("[resolve:" + term + "] kept " + str(len(out)) + " of top candidates")
    return out

EXTRACT_SYSTEM = (
    "Extract from the biomedical question the named entities to look up and the relation(s) it implies.\n"
    'Return ONLY JSON: {"entities": [<entity name>, ...], "predicates": [<predicate>, ...]}.\n'
    "entities: the specific things named in the question (drugs, diseases, genes, ...); [] if none.\n"
    "predicates: choose the relevant relation(s) ONLY from this list, or [] if unconstrained: {predicates}."
)

def retrieve(query):
    system = EXTRACT_SYSTEM.replace("{predicates}", ", ".join(KG2_PREDICATES))
    t0 = time.perf_counter()
    spec = _json(llm(system, query, max_tokens=400))
    print("[llm:extract] " + format(time.perf_counter() - t0, ".1f") + "s -> entities=" + str(spec.get("entities")) + " predicates=" + str(spec.get("predicates")))
    entities = {}
    for term in spec.get("entities", []):
        entities[term] = _search_nodes(term)
    return {"entities": entities, "predicates": spec.get("predicates", [])}

## 3. Select node IDs

In [4]:
def select(retrieval):
    chosen = {}
    for term, df in retrieval["entities"].items():
        if len(df) == 0:
            print("No matches for '" + term + "'.")
            chosen[term] = []
            continue
        if len(df) == 1:
            chosen[term] = [df.iloc[0]["id"]]
            continue
        print("Candidates for '" + term + "' (number, or 'all'):\n")
        for i, row in df.iterrows():
            print("  [" + str(i) + "] " + row["name"][:55] + "  " + row["category"] + "  " + row["id"])
        sel = input("\nNumber, 'all', or blank to skip: ").strip().lower()
        if sel == "all":
            chosen[term] = df["id"].tolist()
        elif sel.isdigit() and int(sel) in df.index:
            chosen[term] = [df.loc[int(sel)]["id"]]
        else:
            chosen[term] = []
    return chosen

## 4. Write the .dl

In [5]:
DL_SYSTEM = (
    "You write Soufflé Datalog to answer a biomedical question over the RTX-KG2 graph.\n"
    "Every program MUST begin with exactly these two lines:\n"
    + EDGE_HEADER + "\n\n"
    'edge(subj, pred, obj, neg) is one KG edge: subj-[pred]->obj are CURIEs; use only neg="False".\n\n'
    "You are given grounded bindings:\n"
    "- Each entity is a SET of equivalent CURIEs. Declare a membership relation and bind the slot "
    "to a variable ranging over the set, e.g.\n"
    "      .decl drug_set(x:symbol)\n"
    '      drug_set("ID1"). drug_set("ID2").\n'
    "  Use the EXACT CURIEs given.\n"
    "- Predicates are exact quoted values.\n\n"
    "Write as many rules as the question needs; multi-hop joins and recursion are allowed. "
    "The final answer relation MUST be named result and hold the CURIEs the question asks for.\n"
    "Output ONLY the program (no prose/markdown). End with .output result. "
    'Use lowercase relation names; avoid the built-in "cat".'
)

def write_dl(query, selection, predicates):
    lines = []
    for term, ids in selection.items():
        if ids:
            idset = ", ".join('"' + i + '"' for i in ids)
            lines.append("entity '" + term + "' = {" + idset + "}")
        else:
            lines.append("entity '" + term + "' = (unresolved)")
    lines.append("predicates = " + (", ".join(predicates) if predicates else "(none specified)"))
    user = "Question: " + query + "\n\nGrounded bindings:\n" + "\n".join(lines) + "\n\nWrite the Soufflé program."
    return _souffle_code(llm(DL_SYSTEM, user, max_tokens=900))

## Pipeline

In [6]:
query = input("Ask a question (e.g. 'what drugs treat parkinson disease?'): ").strip()

_pipe_t0 = time.perf_counter()
r = retrieve(query)
print("Predicates: " + str(r["predicates"]))
for term, df in r["entities"].items():
    print("\nEntity '" + term + "':")
    display(df)

selection = select(r)

_t = time.perf_counter()
program = write_dl(query, selection, r["predicates"])
print("[llm:write_dl] " + format(time.perf_counter() - _t, ".1f") + "s")
print("\n" + program + "\n")

result = run_souffle(program, "result")
if len(result) == 0:
    print("No results.")
    named = result
else:
    ids = result[0].astype(str).tolist()
    namer = "\n".join(
        [
            ".decl node(id:symbol, name:symbol, category:symbol)",
            '.input node(IO=file, filename="' + NODES_TSV + '", columns="7:10:15")',
            ".decl want(id:symbol)",
        ]
        + ['want("' + i + '").' for i in ids]
        + [
            ".decl named(id:symbol, name:symbol, category:symbol)",
            "named(i, n, c) :- node(i, n, c), want(i).",
            ".output named",
        ]
    )
    named = run_souffle(namer, "named")
    if len(named) > 0:
        named.columns = ["id", "name", "category"]
    print(str(len(named)) + " result(s):")
print("[pipeline total] " + format(time.perf_counter() - _pipe_t0, ".1f") + "s")
display(named)

[llm:extract] 2.2s -> entities=['parkinson disease'] predicates=['biolink:treats']
[souffle:cand] wall 72.6s
    compute cand: 39.84s (282 tuples)
    load+overhead: 32.74s
[resolve:parkinson disease] kept 25 of top candidates
Predicates: ['biolink:treats']

Entity 'parkinson disease':


,id,name,category
0,UMLS:C0030567,Parkinson Disease,biolink:Disease
1,MESH:D010300,Parkinson Disease,biolink:Disease
2,OMIM:MTHU074205,Parkinson disease,biolink:NamedThing
3,MONDO:0005180,Parkinson disease,biolink:Disease
4,KEGG:05012,Parkinson disease,biolink:Pathway
5,UMLS:C2931436,Parkinson disease 3,biolink:Disease
6,MESH:C537176,Parkinson disease 3,biolink:Disease
7,UMLS:C3501657,Parkinson Disease 5,biolink:Disease
8,MESH:C566017,Parkinson Disease 5,biolink:Disease
9,UMLS:C3501658,Parkinson Disease 8,biolink:Disease


Candidates for 'parkinson disease' (number, or 'all'):

  [0] Parkinson Disease  biolink:Disease  UMLS:C0030567
  [1] Parkinson Disease  biolink:Disease  MESH:D010300
  [2] Parkinson disease  biolink:NamedThing  OMIM:MTHU074205
  [3] Parkinson disease  biolink:Disease  MONDO:0005180
  [4] Parkinson disease  biolink:Pathway  KEGG:05012
  [5] Parkinson disease 3  biolink:Disease  UMLS:C2931436
  [6] Parkinson disease 3  biolink:Disease  MESH:C537176
  [7] Parkinson Disease 5  biolink:Disease  UMLS:C3501657
  [8] Parkinson Disease 5  biolink:Disease  MESH:C566017
  [9] Parkinson Disease 8  biolink:Disease  UMLS:C3501658
  [10] Parkinson Disease 8  biolink:Disease  MESH:C564631
  [11] Parkinson Disease 13  biolink:Disease  UMLS:C3496588
  [12] Parkinson Disease 13  biolink:Disease  MESH:C565204
  [13] Parkinson disease 10  biolink:Disease  UMLS:C1847360
  [14] Parkinson Disease 10  biolink:Disease  MESH:C564653
  [15] Parkinson Disease 11  biolink:Disease  MESH:C564345
  [16] Parkinson dis

,id,name,category
0,UMLS:C0003779,argipressin,biolink:NamedThing
1,UMLS:C0011701,desmopressin,biolink:NamedThing
2,UMLS:C0037125,Silver,biolink:SmallMolecule
3,UMLS:C0023870,lithium,biolink:ChemicalEntity
4,UMLS:C0009170,cocaine,biolink:ChemicalEntity
...,...,...,...
1599,UMLS:C1328260,"Monitors, Environmental, Chemical Agent",biolink:Device
1600,UMLS:C1537060,Social Security Disability Insurance,biolink:Publication
1601,UMLS:C1562104,Product containing glucagon-like peptide 1 receptor agonist (product),biolink:ChemicalEntity
1602,UMLS:C2917359,GLP-1 Receptor Agonist [EPC],biolink:ChemicalEntity
